In [0]:
# import necessary libraries

from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from pyspark.sql.functions import col, to_date, avg, max as max_s, min as min_s, count

**BASIC PYSPARK**

**Q1**
Read Employees as a DataFrame.

printSchema()

show()

inferSchema=False then manually define schema.


In [0]:
path = 'paste_file_path_here'
employees_df = spark.read\
                    .format('csv')\
                    .option('header','true')\
                    .load(path)

employees_df.printSchema()
employees_df.show()

employees_schema = StructType([StructField('emp_id'    , IntegerType(), nullable = True, metadata = None)
                              ,StructField('name'      , StringType() , nullable = True, metadata = None)
                              ,StructField('dept_id'   , IntegerType(), nullable = True, metadata = None)
                              ,StructField('salary'    , StringType() , nullable = True, metadata = None)
                              ,StructField('city'      , StringType() , nullable = True, metadata = None)
                              ,StructField('join_date' , StringType()   , nullable = True, metadata = None)
                              ,StructField('manager_id', IntegerType(), nullable = True, metadata = None)
                   ])
employees_df = spark.read\
                    .format('csv')\
                    .option('header','true')\
                    .option('inferSchema','false')\
                    .schema(employees_schema)\
                    .load('/Workspace/Users/sumedh.puri@gmail.com/data_engineering/data/raw/employees.csv')
# DateType() for join_date is not able to read the schema and is showing up nulls in the column, it also doesn't any format parameter, its only useful when you use a parquet file as a source
# i am explicitly defining a column with date datatype instead of string, THIS STEP CAN BE DONE LATER though
employees_df = employees_df.withColumn('join_date'
                                       ,to_date(col('join_date'),'dd-MM-yyyy')
                            )
display(employees_df)


------------------------------------------------------------------------------------------------------------------

**Q2**
Select

emp_id
name
salary

only.

In [0]:
employees_df_q2 = employees_df.select('emp_id'
                                     ,'name'
                                     ,'salary'
                               )
display(employees_df_q2)

**Q3**
Filter

Salary > 80,000

In [0]:
employees_df_q3 = employees_df.filter(col("salary")>80000
                               )
display(employees_df_q3)

**Q4**
Find employees

Chicago
AND
salary > 70000

In [0]:
employees_df_q4 = employees_df.filter(  (col('city') == 'Chicago')
                                      & (col('salary') > 70000)
                               )

display(employees_df_q4)

**Q5**
Create

Annual Salary

salary * 12

In [0]:
employees_df_q5 = employees_df.withColumn('annual_salary', col('salary') * 12 
                               )
display(employees_df_q5)

**Q6**
Rename

manager_id to manager

In [0]:
employees_df_q6 = employees_df.withColumnRenamed('manager_id','manager')
display(employees_df_q6)

**AGGREGATIONS**

**Q7**
Department wise

Average salary

In [0]:
employees_df_q7 = employees_df.groupBy(col("dept_id")).agg(avg(col('salary')))

display(employees_df_q7)

**Q8**
Department wise

Maximum salary

Minimum salary

Average salary

Employee count

In [0]:
employees_df_q8 = employees_df.groupBy(col('dept_id')).agg(max_s(col('salary'))
                                                          ,min_s(col('salary'))
                                                          ,avg(col('salary'))
                                                          ,count(col('emp_id'))
                                                       )
display(employees_df_q8)

**Q9**
Find top 3 highest paid employees.

In [0]:
employees_df_q9 = employees_df.select('emp_id').orderBy(col('salary').desc()).limit(3)
display(employees_df_q9)